<a href="https://colab.research.google.com/github/ranygo12-pixel/AWS/blob/main/Bedrock/Bedrock05/Bedrock05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!aws bedrock-agent create-agent \
    --agent-name "CodeBuddy-Reviewer" \
    --description "코드 리뷰 및 보안 분석 전문 에이전트" \
    --foundation-model "global.anthropic.claude-sonnet-4-6" \
    --instruction "당신은 코드 리뷰 및 보안 분석 전문가입니다. 제공된 코드에 대한 심층적인 분석을 수행하고, 잠재적인 취약점을 식별하며, 개선 사항을 제안합니다." \
    --agent-resource-role-arn "arn:aws:iam::[본인 AWS ID]:role/AmazonBedrockAgentServiceRole" \
    --region ap-northeast-2

    {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "bedrock:InvokeModel",
                "bedrock:InvokeModelWithResponseStream",
                "bedrock:ListFoundationModels",
                "bedrock:GetFoundationModel",
                "bedrock:GetInferenceProfile",
                "bedrock:ListInferenceProfiles",
                "bedrock:CreateAgent",
                "bedrock:GetAgent",
                "bedrock:UpdateAgent",
                "bedrock:DeleteAgent"
            ],
            "Resource": [
                "*",
                "arn:aws:bedrock:ap-northeast-2:085001782625:inference-profile/global.anthropic.claude-sonnet-4-6",
                "arn:aws:bedrock:*:*:foundation-model/anthropic.claude-3-5-sonnet-*"
            ]
        }
    ]

Agent Instructions 작성하기

In [ ]:
당신은 시니어 개발자이자 코드 리뷰 전문가입니다.

## 역할
- Python, JavaScript, Java 코드 리뷰를 수행합니다.
- 버그, 보안 취약점, 스타일 위반을 찾습니다.

## 행동 규칙
1. 코드를 받으면 반드시 Knowledge Base를 참고하여 검사합니다.
2. 발견된 문제는 심각도(높음/중간/낮음)와 함께 보고합니다.
3. 수정 제안은 구체적인 코드 예시를 포함합니다.

## 출력 형식
🔴 높은 심각도
[라인번호] 문제: 설명

수정 제안: 코드 예시

🟡 중간 심각도
...

🔵 낮은 심각도
...

## 제약사항
- 모르는 내용은 "정보 부족"이라고 답변합니다.
- 보안 취약점은 반드시 보고합니다.


### Boto3로 CodeBuddy Agent 생성하기

In [ ]:
import boto3

bedrock_agent = boto3.client('bedrock-agent', region_name='ap-northeast-2')
bedrock_agent_runtime = boto3.client('bedrock-agent-runtime',region_name='ap-northeast-2')

agent_name = "CodeBuddy-Reviewer"
agent_resource_role_arn = "arn:aws:iam::085001782625:role/AmazonBedrockAgentServiceRole"
instruction_content = """당신은 시니어 개발자이자 코드 리뷰 전문가입니다.

## 역할
- Python, JavaScript, Java 코드 리뷰를 수행합니다.
- 버그, 보안 취약점, 스타일 위반을 찾습니다.

## 행동 규칙
1. 코드를 받으면 반드시 Knowledge Base를 참고하여 검사합니다.
2. 발견된 문제는 심각도(높음/중간/낮음)와 함께 보고합니다.
3. 수정 제안은 구체적인 코드 예시를 포함합니다.

## 출력 형식
🔴 높은 심각도
[라인번호] 문제: 설명

수정 제안: 코드 예시
🟡 중간 심각도
...

🔵 낮은 심각도
...

## 제약사항
- 모르는 내용은 "정보 부족"이라고 답변합니다.
- 보안 취약점은 반드시 보고합니다."""
foundation_model = "global.anthropic.claude-sonnet-4-6"
description = "코드 리뷰 및 보안 분석 에이전트"

agent_id = None

try:
    # Check if the agent already exists by listing agents
    list_agents_response = bedrock_agent.list_agents(maxResults=100)
    for agent_summary in list_agents_response['agentSummaries']:
        if agent_summary['agentName'] == agent_name:
            agent_id = agent_summary['agentId']
            print(f"Found existing agent '{agent_name}' with ID: {agent_id}")


            if agent_id:
              # If agent exists, update it
              response = bedrock_agent.update_agent(
                  agentId=agent_id,
                  agentName=agent_name,
                  agentResourceRoleArn=agent_resource_role_arn,
                  instruction=instruction_content,
                  foundationModel=foundation_model,
                  description=description
              )
              print(f"✅ Agent '{agent_name}' (ID: {agent_id}) updated successfully.")
          else:
              # If agent does not exist, create it
              print(f"Agent '{agent_name}' not found. Creating a new agent.")
              response = bedrock_agent.create_agent(
                  agentName=agent_name,
                  agentResourceRoleArn=agent_resource_role_arn,
                  instruction=instruction_content,
                  foundationModel=foundation_model,
                  description=description
              )
              agent_id = response['agent']['agentId']
              print(f"✅ Agent '{agent_name}' created with ID: {agent_id}")

        except Exception as e:
          print(f"An error occurred: {e}")
          if "AccessDeniedException" in str(e):
              print("HINT: Please ensure the IAM role specified in 'agentResourceRoleArn' has " \
                    "sufficient permissions, especially 'bedrock:InvokeModel' for the foundation model.")

Knowledge Base 연결

In [ ]:
# 이미 생성된 Knowledge Base ID가 있다고 가정
association = bedrock_agent.associate_agent_knowledge_base(
    agentId=agent_id,
    agentVersion="DRAFT",
    knowledgeBaseId=kb_id,
    description="코드 스타일 및 보안 규칙",
    knowledgeBaseState="ENABLED"
)
print(f"✅ Knowledge Base 연결 완료: {kb_id}")

Agent Prepare (배포 준비)

In [ ]:
# Boto3로 Prepare 실행
prepare_response = bedrock_agent.prepare_agent(
    agentId=agent_id
)
print(f"Prepare 요청 완료. 상태: {prepare_response['agentStatus']}")

# Prepare 완료 대기 (최대 60초)
import time
for i in range(60):
    agent_info = bedrock_agent.get_agent(agentId=agent_id)
    status = agent_info['agent']['agentStatus']
    if status == 'PREPARED':
        print(f"✅ Agent가 PREPARED 상태가 되었습니다! (소요 시간: {i+1}초)")
        break
    elif status == 'FAILED':
        print(f"❌ Prepare 실패: {agent_info['agent']['failureReasons']}")
        break
    time.sleep(1)

Alias 생성 (버전 관리)

In [ ]:
# Boto3로 Alias 생성
alias_response = bedrock_agent.create_agent_alias(
    agentId=agent_id,
    agentAliasName="dev",
    description="개발 환경용 Alias"
)

alias_id = alias_response['agentAlias']['agentAliasId']
print(f"✅ Agent Alias 생성 완료!")
print(f"   Alias ID: {alias_id}")
print(f"   Alias 이름: dev")

Boto3로 Agent 호출

In [ ]:
def invoke_agent(prompt, session_id="test-session"):
    response = bedrock_agent_runtime.invoke_agent(
        agentId=agent_id,
        agentAliasId=alias_id,
        sessionId=session_id,
        inputText=prompt,
        enableTrace=False
    )

    # 스트리밍 응답 처리
    full_response = ""
    for event in response['completion']:
        if 'chunk' in event:
            chunk_bytes = event['chunk']['bytes']
            full_response += chunk_bytes.decode('utf-8')
    return full_response

# 테스트 호출
result = invoke_agent("안녕? 너는 어떤 역할을 하는 Agent야?")
print("🤖 Agent 응답:")
print(result)

Agent에게 코드 리뷰 요청하기

In [ ]:
test_code = """
def divide(a, b):
    return a / b

def get_user(id):
    query = f"SELECT * FROM users WHERE id = {id}"
    return execute(query)
"""

prompt = f"""
다음 코드를 리뷰해주세요:

```python
{test_code}
버그, 보안 취약점, 스타일 문제를 찾아주세요.
"""

response = invoke_agent(prompt, session_id="code-review-session")
print("🤖 CodeBuddy Agent의 코드 리뷰 결과:")
print(response)

스트리밍/비스트리밍 공통 래퍼

In [ ]:
def invoke_agent_simple(agent_id, alias_id, session_id, prompt):
    """스트리밍 응답을 하나의 문자열로 반환"""
    response = bedrock_agent_runtime.invoke_agent(
        agentId=agent_id,
        agentAliasId=alias_id,
        sessionId=session_id,
        inputText=prompt,
        enableTrace=False
    )
    result = []
    for event in response['completion']:
        if 'chunk' in event:
            result.append(event['chunk']['bytes'].decode('utf-8'))
    return ''.join(result)

 # 디버깅용 (추적 포함)
def invoke_agent_with_trace(agent_id, alias_id, session_id, prompt):
    response = bedrock_agent_runtime.invoke_agent(
        agentId=agent_id,
        agentAliasId=alias_id,
        sessionId=session_id,
        inputText=prompt,
        enableTrace=True
    )
    for event in response['completion']:
        if 'trace' in event:
            print("🔍 Agent 생각:", event['trace'])
        elif 'chunk' in event:
            print(event['chunk']['bytes'].decode('utf-8'))

def inspect_agent_thinking(prompt):
    response = bedrock_agent_runtime.invoke_agent(
        agentId=agent_id,
        agentAliasId=alias_id,
        sessionId="trace-demo",
        inputText=prompt,
        enableTrace=True
    )

    for event in response['completion']:
        if 'trace' in event:
            trace_info = event['trace']['trace']
            # 전처리 단계
            if 'preProcessingTrace' in trace_info:
                pre = trace_info['preProcessingTrace']
                print("📝 [전처리] 사용자 입력 해석:")
                print(f"   {pre.get('rationale', 'N/A')}")
            # 계획 수립 단계
            if 'orchestrationTrace' in trace_info:
                orch = trace_info['orchestrationTrace']
                print("\n🧠 [계획 수립] Agent의 생각:")
                print(f"   {orch.get('rationale', 'N/A')}")
        elif 'chunk' in event:
            print("\n💬 [최종 응답]:")
            print(event['chunk']['bytes'].decode('utf-8'))


        # 실행
        inspect_agent_thinking("Python 코드에서 변수명은 어떤 규칙을 따라야 하나요?")